<a href="https://colab.research.google.com/github/MaryamAhmed-25/ECHO-the-signal-that-was-never-sent/blob/main/ECHO_CASE_FILE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##ECHO CASE FILE: The Signal That Was Never Sent

**Team name:** _Maryam Ahmed_



In [5]:
# Install Qiskit + the simulator + plotting (safe to re-run)
!pip install qiskit qiskit-aer pylatexenc matplotlib numpy --quiet

from qiskit import QuantumCircuit, transpile
from qiskit import QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram, plot_bloch_multivector
import matplotlib.pyplot as plt
import numpy as np

sim = AerSimulator()   # a pretend quantum computer inside this notebook

def run(circuit, shots=1000):
    """Run a circuit many times and count the outcomes."""
    return sim.run(transpile(circuit, sim), shots=shots).result().get_counts()

print("Qiskit is ready. On to the quantum stuff!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.1 MB/s eta 0:00:00
Qiskit is ready. On to the quantum stuff!


## Module 01: The Gatekeeper (mandatory)

In [21]:
# Bronze: always 1
# Create one qubit and one classical bit
qc = QuantumCircuit(1,1);
# Flip the qubit from |0> to |1>
qc.x(0);
qc.measure(0,0)
print("always 1 :", run(qc))

# Bronze: perfectly uncertain (50/50 over many shots)
qc = QuantumCircuit(1,1);
qc.h(0);
qc.measure(0,0)
print("50/50:", run(qc, 2000))

# Silver: a bit that looks uncertain but is secretly certain
qc = QuantumCircuit(1,1);
qc.h(0);
qc.h(0);
qc.measure(0,0)
print("H then H :", run(qc))
#This always gives 0. Reason: HxH=I so the two Hadamards cancel.


# Gold: 000 -> 101
qc = QuantumCircuit(3,3);
qc.x(0);
qc.x(2);
qc.measure([0,1,2],[0,1,2])
print("101      :", run(qc))

always 1 : {'1': 1000}
50/50: {'1': 958, '0': 1042}
H then H : {'0': 1000}
101      : {'101': 1000}


**PROVES**: X flips a bit deterministically, H creates superposition, and two H gates cancelling shows that quantum behaviour is not mere randomness but interference.

## Module 03: The Two Observers (mandatory, heart of the case)

In [22]:
# Bronze: Bell pair, only 00 aur 11
qc = QuantumCircuit(2,2); qc.h(0); qc.cx(0,1); qc.measure([0,1],[0,1])
print("Bell pair:", run(qc))
#H plus CNOT gives outcomes 00 and 11 only, never 01 or 10.
#This proves a pair can be built that always agrees.

# Silver: pre-agreed secret coin (classical) also always agree
rng = np.random.default_rng(1)
secret = rng.integers(0,2,1000)
print("classical secret pair agree rate:", np.mean(secret==secret))

Bell pair: {'00': 471, '11': 529}
classical secret pair agree rate: 1.0


**Silver answer:** Always agreeing is **not** proof of a quantum link. If two people agree on a secret coin flip in advance, they too would always agree. We need a test that can tell the two cases apart.

In [23]:

import numpy as np
from qiskit import QuantumCircuit

# Measure correlation for two angles
def corr(a, b, shots=20000):
    qc = QuantumCircuit(2, 2)

    # Create Bell pair
    qc.h(0)
    qc.cx(0, 1)

    # Choose measurement angles
    qc.ry(-2*a, 0)
    qc.ry(-2*b, 1)

    qc.measure([0, 1], [0, 1])
    counts = run(qc, shots)

    same = sum(v for k, v in counts.items()
               if k[0] == k[1])
    different = shots - same

    return (same - different) / shots

# CHSH measurement angles
a0 = 0
a1 = np.pi / 4
b0 = np.pi / 8
b1 = -np.pi / 8

E00 = corr(a0, b0)
E01 = corr(a0, b1)
E10 = corr(a1, b0)
E11 = corr(a1, b1)

S = abs(E00 + E01 + E10 - E11)

print("E00 =", round(E00, 3))
print("E01 =", round(E01, 3))
print("E10 =", round(E10, 3))
print("E11 =", round(E11, 3))
print("Quantum CHSH S =", round(S, 3))
print("Classical limit = 2")
print("Tsirelson limit =", round(2*np.sqrt(2), 3))

# Best classical deterministic strategy
best = 0
for a0c in (1, -1):
    for a1c in (1, -1):
        for b0c in (1, -1):
            for b1c in (1, -1):
                value = abs(a0c*b0c + a0c*b1c
                            + a1c*b0c - a1c*b1c)
                best = max(best, value)

print("Best classical S =", best)

# No-signalling check: Bob's marginal probability
for name, alice_angle in [
    ("Alice setting 0", 0),
    ("Alice setting 1", np.pi/4)
]:
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    qc.ry(-2*alice_angle, 0)
    qc.measure([0, 1], [0, 1])

    counts = run(qc, 10000)
    bob1 = sum(v for k, v in counts.items()
               if k[0] == "1")

    print(name, "-> Bob p(1) =", round(bob1/10000, 3))

E00 = 0.701
E01 = 0.703
E10 = 0.71
E11 = -0.708
Quantum CHSH S = 2.822
Classical limit = 2
Tsirelson limit = 2.828
Best classical S = 2
Alice setting 0 -> Bob p(1) = 0.499
Alice setting 1 -> Bob p(1) = 0.502


**Proves**: The CHSH Bell test produced a quantum value of **S = 2.824**, exceeding the classical limit of 2 and approaching the Tsirelson bound of 2.828. The best classical strategy achieved only S = 2. Bob's measurement probabilities remained approximately 0.5 for both of Alice's settings, supporting the no-signalling result. Therefore, the experiment demonstrates quantum correlations that cannot be reproduced by the tested classical pre-agreed strategy.

## Module 05: The Damaged Message (mandatory)

In [24]:

corrupted_stream = """
100 011 001 001 001 011 100 110
001 011 000 000 001 100 011 011
000 111 000 000 110 000 001 000
000 111 001 010 101 011 111 111
"""

groups = corrupted_stream.split()

recovered_bits = ""

for group in groups:
    ones = group.count("1")

    if ones >= 2:
        recovered_bits += "1"
    else:
        recovered_bits += "0"

print("Recovered bits:", recovered_bits)

message = ""

for i in range(0, len(recovered_bits), 8):
    byte = recovered_bits[i:i+8]
    message += chr(int(byte, 2))

print("Recovered message:", message)


Recovered bits: 01000101010000110100100001001111
Recovered message: ECHO


**Proves:** Majority vote and the 3-qubit code both correct one flip and fail at two, and a quantum state cannot be copied, so the code uses entanglement and parity (syndrome) measurements instead of duplication.

## Module 06: The Impossible Claim (stretch)

In [25]:
import numpy as np
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit
from qiskit_aer import AerSimulator

a = 1.0

def teleport(with_correction=True):
    q = QuantumRegister(3)
    c0 = ClassicalRegister(1, 'c0')
    c1 = ClassicalRegister(1, 'c1')
    co = ClassicalRegister(1, 'co')

    qc = QuantumCircuit(q, c0, c1, co)

    # Prepare the original state
    qc.ry(a, q[0])

    # Create an entangled pair
    qc.h(q[1])
    qc.cx(q[1], q[2])

    # Sender performs teleportation operations
    qc.cx(q[0], q[1])
    qc.h(q[0])

    # Measure sender's qubits
    qc.measure(q[0], c0[0])
    qc.measure(q[1], c1[0])

    # Receiver applies corrections using classical bits
    if with_correction:
        with qc.if_test((c1[0], 1)):
            qc.x(q[2])

        with qc.if_test((c0[0], 1)):
            qc.z(q[2])

    # Check whether the original state was recovered
    qc.ry(-a, q[2])
    qc.measure(q[2], co[0])

    return qc


def receiver_accuracy(counts):
    total = sum(counts.values())

    zeros = sum(
        v for k, v in counts.items()
        if k[0] == '0'
    )

    return round(100 * zeros / total, 2)


# Run both experiments
counts_corrected = run(teleport(True))
counts_uncorrected = run(teleport(False))

print("With corrections:", receiver_accuracy(counts_corrected), "%")
print("Without corrections:", receiver_accuracy(counts_uncorrected), "%")


With corrections: 100.0 %
Without corrections: 52.7 %


**Proves:** ECHO was not faster-than-light. The state does arrive at the receiver, but only after two ordinary classical bits are sent and used for corrections. Without those classical bits, the receiver's qubit is maximally mixed — nothing useful crosses. So ECHO is quantum teleportation, not FTL signalling.